In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-4: Case study: GRPO on a real run

**RL for LLMs, session 4.**


## What this session teaches

One documented run shows GRPO at work. DeepSeek-R1-Zero trained a model on math problems with RL alone.

Its reward came from a **verifier**, a program that checks the final answer, which must sit in a fixed format. No reward model is learned, so there is no learned estimate to exploit.

GRPO turns rewards into weights. For each question it samples a group of answers. An answer's **advantage** is its reward minus the group's average, divided by the group's spread. Every token of that answer is pushed by that one number.

The group replaces a critic: its average says what was expected. Two consequences follow. If every answer in a group gets the same reward, every advantage is zero and nothing is learned. And every token shares its answer's number, good steps and bad. <!-- long-ok -->

RL can only reinforce what the model samples. If no answer is ever right, every group is all-equal. SFT first teaches the format, so a right answer becomes likely enough to reinforce. R1-Zero's large model managed without; a small one may not. <!-- long-ok -->

The question to carry: what does a group give GRPO in place of a critic, and when does it give nothing?

By the end you can

- say how GRPO turns a group's rewards into one weight per answer
- say when a group teaches nothing, and why a verifier needs a format
- say why SFT comes before RL

**Where this session fits: the topics of the series, in order**

<img src="images/rllm4v2_strip.png" width=88%>

- this session starts topic 4: a verifier grades, and GRPO runs on a real task


## Setting and words

**Why this slide?** The opening uses these words before the slides show them; each is said by what it does.

| word | what it means here |
|:--|:--|
| **answer** (also a **completion**) | the text the model writes for a prompt |
| **R1-Zero** and **R1** | the two models of the run: R1-Zero is trained by RL alone on a pretrained base model; R1 is trained with SFT first, then RL |
| **group** | the answers sampled for one question, scored together |
| **advantage** (here, the **weight**) | the number that pushes an answer up (positive) or down (negative); GRPO's weight on every token of an answer |
| **critic** | a network that predicts the return still to come (Intro B3); GRPO has none |
| **verifier** and **reward model** | a verifier is a program that scores an answer; a reward model is a trained network. R1-Zero used only a verifier |
| **reward = return** | the reward comes once, at the end, so an answer's return equals its reward |


## The map: four sections

**Why this slide?** The opening has several ideas; this is the order they come in, with the question each answers.

<img src="images/rllm4v2_map.png" width=96%>

- **R1-Zero** (RL alone) is the run the first three sections use
- **R1** (SFT first, then RL) comes in the last section, as the way to make a right answer reachable
- section 3 holds the two consequences of a group-relative weight: consequence 1 (an all-equal group) and consequence 2 (one weight per answer)
- the four boxes above reappear at the start of each section, with the current section outlined


## What a verifier can reward

<img src="images/rllm4v2_bar_1.png" width=70%>

**You are here:** section 1 of 4

**Why this section?** GRPO needs a reward for every answer. Here a program, not a trained network, gives it.

> **R1-Zero was trained with RL alone on math problems; a verifier scores each answer it writes.**

<img src="images/rllm4v2_loop.png" width=52%>

- a pretrained base model starts, with no SFT first (**fact**, [DeepSeek-R1 paper](https://arxiv.org/abs/2501.12948))
- each round: sample a group of answers to a question, score each with the verifier, compute advantages, update
- the answer comes after a **trace**: the model's written reasoning, with checking and revising


## Accuracy reward and format reward

**Why this slide?** The verifier must find the answer in the text. That is why a format is part of the reward.

> **RLVR (RL with verifiable rewards, from RLLM-1): the reward is a program's check, so there is no reward model to train and none to exploit.**

<img src="images/rllm4v2_format.png" width=58%>

- the **accuracy reward** is 1 if the final answer matches the key
    - the answer must sit in a fixed place, so the program can find it
- the **format reward** is 1 if the `<think>` and `</think>` and `<answer>` and `</answer>` markers are present and in order
- the R1 paper uses rule-based rewards and no neural reward model (**fact**)
    - its reason: neural reward models are open to reward hacking in large-scale RL
- a rule can still be fooled: empty markers earn the format reward (**our interpretation**)
- the training prompt states the markers: Table 1 of the paper tells the model to write `<think>` and `<answer>` (**fact**)
- so the format reward rests on the base model following that instruction at least sometimes (**our interpretation**)


## Who grades, and what is graded

**Why this slide?** Three questions are easy to mix: what is graded, who grades, and how many numbers reach GRPO.

> **A program can give credit for steps too. Either way, GRPO receives one reward per answer, unless a trained network judges each step.**

| | R1-Zero's reward | a verifier with partial credit for steps | a process reward model (PRM) |
|:--|:--|:--|:--|
| what is graded | the final answer, and the format | the final answer, and whether each step is well formed | each step of the reasoning |
| who grades | a verifier (a program) | a verifier (a program) | a trained network |
| rewards reaching GRPO | one per answer | one per answer (the credits are summed) | one per step |

- an **outcome reward model** (ORM) is a trained network that judges the finished answer
- R1-Zero used neither an ORM nor a PRM
- the R1 paper lists a PRM among its unsuccessful attempts (**fact**)
    - its reasons: a step is hard to define, hard to check, and a trained PRM can be hacked
- DeepSeekMath's process-supervision variant gives each token the summed step rewards that follow it (**fact**)


## From rewards to weights

<img src="images/rllm4v2_bar_2.png" width=70%>

**You are here:** section 2 of 4

**Why this section?** Rewards are numbers on answers. The update multiplies each token's gradient by a weight. GRPO's weight is the advantage.

> **For each question, sample a group of answers. An answer's advantage is its reward minus the group mean, divided by the group's spread. Every token of that answer is pushed by it.**

<img src="images/rllm4v2_group.png" width=40%>

- a positive advantage pushes the answer up, a negative one pushes it down
- the group's mean plays the critic's part: it says what to expect
    - a critic is a network to train; the mean needs no training
- the formula is Intro C2's: $A_i = \dfrac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r)}$ over the group


## One worked group

**Why this slide?** This is the one place to compute an advantage by hand. The numbers are made up, on a made-up task: scale a recipe, then convert its units.

> **Rewards are centered and divided by the spread within the group, so an answer is judged against its siblings, not against a fixed value.**

- one question, $G = 5$ answers
- each reward adds partial credit for steps to credit for the answer
| answer | reward | reward minus mean | advantage |
|:--|:--|:--|:--|
| 1 | 1.0 | +0.36 | +1.01 |
| 2 | 0.4 | -0.24 | -0.67 |
| 3 | 0.2 | -0.44 | -1.23 |
| 4 | 1.0 | +0.36 | +1.01 |
| 5 | 0.6 | -0.04 | -0.11 |

- the mean is $0.64$ and the standard deviation is $0.36$ (divisor $G - 1$, as in Intro C2)
- the advantages add up to $0$: pushing some answers up always pushes others down
- all rewards are positive, yet two advantages are negative: the comparison is relative


## Consequence 1: an all-equal group teaches nothing

<img src="images/rllm4v2_bar_3.png" width=70%>

**You are here:** section 3 of 4

**Why this section?** A group-relative weight has a cost: when the group cannot tell answers apart, there is nothing to learn from.

> **If every answer in a group gets the same reward, the spread is zero and every advantage is zero. Nothing is learned from that question.**
<!-- seed-plant: all-equal-group -->

<img src="images/rllm4v2_equal.png" width=46%>

- with a 0-or-1 reward, all-wrong and all-right groups are both all-equal
- if no answer is ever right, every group is all-equal: the model learns nothing, however long it trains
- RLLM-6 returns to this with a method that filters such groups out
- the fix this session shows for a model that is never right comes in the last section


## Consequence 2: every token shares one number

**Why this slide?** The group gives one weight per answer. What does that do to the single steps inside it?

> **Every token of an answer gets the same advantage, the good steps and the bad ones. The group cannot say which step earned the reward.**

<img src="images/rllm4v2_tokens.png" width=64%>

- a right answer with one wrong step pushes the wrong step up as much as the rest (**our interpretation**)
- giving partial credit for steps changes the reward, but it still reaches GRPO as one reward per answer
- a critic could give each token its own credit
- GRPO dropped the critic and pays this price
- RLLM-5 shows what per-token credit would buy


## R1-Zero: RL reinforces only what the model samples

<img src="images/rllm4v2_bar_4.png" width=70%>

**You are here:** section 4 of 4

**Why this section?** An advantage can only push up answers that were sampled. Whether RL can start depends on the base model.

> **RL reinforces only what the model samples. R1-Zero's large base model was sometimes right from the start; a smaller one may never be.**

<img src="images/rllm5_results.png" width=32%>

- the revised paper (arXiv v2) reports R1-Zero's AIME 2024 **pass@1** (the share of problems one sample solves) rising from 15.6% to 77.9% (**fact**)
- **majority voting** (sample many answers, take the most common) reaches 86.7%: a different measure, not comparable to pass@1
- the same paper reports that smaller models relying on large-scale RL "may not even achieve the performance of distillation" (**fact**)
- from that, a small model that is never right is plausible (**our interpretation**)


## R1: making a right answer reachable

**Why this slide?** If no answer is right, every group is all-equal. Three ways make a right answer likely enough to reinforce, shown on a made-up multi-step task.

> **Reachable first, right second: each way gives groups some spread before any answer is fully right. This staging is curriculum-like (our interpretation).**

| way | what it makes likely | when it helps | its limit |
|:--|:--|:--|:--|
| **SFT first** (R1) | the format and a readable style | the base model never writes it | needs examples; does not make the answer right |
| **format reward** (R1-Zero) | the markers, so a verifier can read the answer | the prompt states the format and the model follows it at least sometimes | pays for form, not correctness; earns nothing if the prompt is ignored |
| **partial credit for steps** | well-formed steps | answers are not yet right | can be earned without a right answer |

- the R1 paper states two reasons for its cold-start SFT: poor readability and language mixing in R1-Zero (**fact**)
- credit for form keeps paying once the form is learned, so it protects the format (**our interpretation**)
- once form is reliable it pushes little toward right values
    - credit for the answer itself does that (**our interpretation**)
- all three still reach GRPO as one reward per answer


# What we established, and what it sets up

- **Established**
    - GRPO turns a group of rewards into one weight per answer: reward minus the group mean, over the spread
    - the group replaces a critic, at two costs: an all-equal group teaches nothing, and every token shares its answer's number
    - R1-Zero used a verifier with accuracy and format rewards, which needs a fixed answer format
    - RL reinforces only what it samples, so SFT first (R1), a format reward or partial credit for steps makes success reachable
- **Sets up**
    - RLLM-5: what per-token credit would buy over one weight per answer
    - companion sessions: **P1_Preflight** before a first GPU run, **P1_ReadingARun** while a run is going
